# Complete Colab hyperparameter search

Primary user-facing notebook for selecting hyperparameters. It keeps search-space validation, calibration, estimation, execution, resumption, Pareto analysis, benchmark comparison, and export modular by calling `hpo` package APIs.

## 1. Setup

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json
REPO_URL="https://github.com/TrueRottweiler/WashingtonCsed504.git"; BRANCH="feature/hpo-framework"; REPO_ROOT=Path("/content/WashingtonCsed504")
if not (REPO_ROOT/"src/a1-cv/hpo").exists():
    if REPO_ROOT.exists(): shutil.rmtree(REPO_ROOT)
    subprocess.run(["git","clone","--branch",BRANCH,"--single-branch",REPO_URL,str(REPO_ROOT)],check=True)
CV_DIR=REPO_ROOT/"src/a1-cv"; os.chdir(CV_DIR); sys.path.insert(0,str(CV_DIR)) if str(CV_DIR) not in sys.path else None
subprocess.run([sys.executable,"-m","pip","install","-q","-r","hpo_requirements.txt"],check=True)
import torch, optuna, pandas as pd
print(sys.version); print(torch.__version__, torch.version.cuda); print(optuna.__version__)

## 2. Persistence and Google Drive

In [ ]:
MOUNT_DRIVE=False
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST_ROOT=Path("/content/drive/MyDrive/WashingtonCsed504-HPO")
else:
    PERSIST_ROOT=Path("/content/WashingtonCsed504-HPO")
PERSIST_ROOT.mkdir(parents=True,exist_ok=True)
RESUME=True
print("Persistence root:",PERSIST_ROOT)

## 3. Hardware profile, recommendations, and overrides

In [ ]:
from hpo.hardware import detect_hardware
from hpo.scheduler import plan_resources
hardware=detect_hardware(); print(json.dumps(hardware.to_dict(),indent=2,default=str))
DEVICE_OVERRIDE="auto"; CONCURRENT_TRIALS_OVERRIDE=None; INTRAOP_THREADS_OVERRIDE=None; INTEROP_THREADS_OVERRIDE=None; WORKERS_OVERRIDE=None; MEMORY_RESERVE_GB=1.0
resource_plan=plan_resources(hardware,device=DEVICE_OVERRIDE,requested_concurrency=CONCURRENT_TRIALS_OVERRIDE,requested_intraop_threads=INTRAOP_THREADS_OVERRIDE,requested_interop_threads=INTEROP_THREADS_OVERRIDE,requested_workers=WORKERS_OVERRIDE,memory_reserve_gb=MEMORY_RESERVE_GB)
print(json.dumps(resource_plan.to_dict(),indent=2))

## 4. Experiment selection

In [ ]:
DATASET="cifar10"  # cifar10, cifar100, imagenet32
MODEL="resnet18"   # resnet18, resnet50, vit, vit_base
if DATASET=="imagenet32":
    IMAGENET32_ROOT="/content/imagenet32"  # must already contain the repository-supported data
else: IMAGENET32_ROOT=None
NUM_CLASSES={"cifar10":10,"cifar100":100,"imagenet32":1000}[DATASET]
print(DATASET,MODEL,NUM_CLASSES)

## 5. Search-space source: built-in, Python dictionary, Python list, CSV upload, JSON/YAML, or manual input

In [ ]:
from hpo.search_space import normalize_space, load_csv, load_json, load_yaml, preview_rows, combination_count
from hpo.notebook_api import preview_dataframe, optional_widgets
INPUT_SOURCE="builtin"  # builtin, dictionary, list, csv, csv_upload, json, yaml, manual
BUILTIN_CSV=CV_DIR/("hpo_configs/search_spaces/vit_cifar.csv" if MODEL.startswith("vit") else "hpo_configs/search_spaces/resnet18_cifar.csv")
DICT_SPACE={"learning_rate":{"type":"float","low":1e-4,"high":0.2,"log":True,"default":0.01},"batch_size":{"type":"categorical","choices":[64,128,256],"default":128},"optimizer":{"type":"categorical","choices":["sgd","adamw"],"default":"sgd"},"momentum":{"type":"float","low":0.8,"high":0.95,"step":0.05,"default":0.9,"condition":'optimizer == "sgd"'},"beta1":{"type":"float","low":0.8,"high":0.95,"step":0.05,"default":0.9,"condition":'optimizer == "adamw"'}}
LIST_SPACE=[{"name":name,**value} for name,value in DICT_SPACE.items()]
MANUAL_SPACE=DICT_SPACE.copy()
if INPUT_SOURCE=="builtin" or INPUT_SOURCE=="csv":
    specs=load_csv(BUILTIN_CSV)
elif INPUT_SOURCE=="csv_upload":
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("CSV upload control is available in Colab; use INPUT_SOURCE='csv' with a local path elsewhere") from exc
    uploaded=files.upload()
    if len(uploaded)!=1:
        raise ValueError("Upload exactly one CSV search-space file")
    from hpo.notebook_api import normalize_uploaded_csv
    filename,content=next(iter(uploaded.items()))
    specs=normalize_uploaded_csv(content,filename=filename)
elif INPUT_SOURCE=="dictionary": specs=normalize_space(DICT_SPACE,source_name="dictionary cell")
elif INPUT_SOURCE=="list": specs=normalize_space(LIST_SPACE,source_name="list cell")
elif INPUT_SOURCE=="manual": specs=normalize_space(MANUAL_SPACE,source_name="manual cell")
elif INPUT_SOURCE=="json": specs=load_json(Path("/content/search_space.json"))
elif INPUT_SOURCE=="yaml": specs=load_yaml(Path("/content/search_space.yaml"))
else: raise ValueError(INPUT_SOURCE)
display(preview_dataframe(specs)); print("Finite combinations:",combination_count(specs)); display(optional_widgets() or "ipywidgets unavailable")

## 6. Validation and feasibility preview

In [ ]:
from hpo.constraints import validate_candidate
from hpo.exceptions import InvalidTrialError
warnings=[]
for spec in specs:
    if spec.name=="batch_size": warnings.append("Batch values will be filtered by calibration before launch.")
print("Conditions:",[s.condition for s in specs if s.condition])
print("Defaults:",{s.name:s.default for s in specs if s.default is not None})
print("Warnings:",warnings)
# Structural model constraints are checked before allocation: ViT divisibility/patch size, optimizer-specific fields, precision, and memory calibration.

## 7. Search mode and continuous execution

In [ ]:
MODE="successive_halving"  # proxy, successive_halving, full
CONTINUOUS=False
CONTINUOUS_SETTINGS={"enabled":CONTINUOUS,"strategy":MODE,"maximum_trials":None,"maximum_wall_time_hours":8.0,"maximum_gpu_hours":None,"maximum_cpu_hours":None,"maximum_cost_usd":None,"target_validation_metric":None,"stop_after_no_improvement_trials":25,"minimum_improvement":0.0005,"pareto_stagnation_trials":25,"checkpoint_after_each_trial":True}
print(MODE,json.dumps(CONTINUOUS_SETTINGS,indent=2))

## 8. Objectives, hard constraints, and cost rates

In [ ]:
OBJECTIVES=[{"name":"validation_top1","direction":"maximize","primary":True},{"name":"wall_seconds","direction":"minimize","primary":False},{"name":"peak_gpu_memory_mb","direction":"minimize","primary":False}]
CONSTRAINTS=[]  # e.g. {"name":"peak_gpu_memory_mb","operator":"<=","value":12000}
COST_RATES={"gpu_usd_per_hour":None,"cpu_usd_per_hour":None,"storage_usd_per_gb_month":None,"electricity_usd_per_kwh":None,"colab_subscription_usd":None,"colab_compute_unit_usd":None}
print(json.dumps({"objectives":OBJECTIVES,"constraints":CONSTRAINTS,"rates":COST_RATES},indent=2))

## 9. Calibration: batch size, real training steps, validation, checkpoint, and memory

In [ ]:
from hpo.adapters import RepoModules, build_trial_model, build_trial_dataset
from hpo.calibration import calibrate_batch_sizes
modules=RepoModules(REPO_ROOT); device=torch.device(resource_plan.device)
dataset_cfg={"name":DATASET,"validation_fraction":0.1,"split_seed":42}; dataset_cfg.update({"root":IMAGENET32_ROOT} if IMAGENET32_ROOT else {})
bundle=build_trial_dataset(modules,dataset_cfg,{"seed":42},device)
model_cfg={"name":MODEL}
def make_batch(bs):
    x,y=next(bundle.train.epoch(bs,train=True)); return x,y
calibration=calibrate_batch_sizes(lambda: build_trial_model(modules,model_cfg,bundle.num_classes,device),make_batch,device=device,candidates=[32,64,128,256,512],precision="bf16" if hardware.bf16_native else "fp16" if device.type=="cuda" else "fp32",warmup_steps=2,measure_steps=3,channels_last=MODEL.startswith("resnet"))
print(json.dumps(calibration.to_dict(),indent=2))

## 10. Build configuration and pre-search estimate

In [ ]:
import yaml, copy
from hpo.config import load_study_config
from hpo.estimation import estimate_search
STUDY_NAME=f"{MODEL}-{DATASET}-{MODE}"
CONFIG_PATH=PERSIST_ROOT/f"{STUDY_NAME}.yaml"
base=yaml.safe_load((CV_DIR/("hpo_configs/colab/vit_cifar10.yaml" if MODEL.startswith("vit") else "hpo_configs/colab/resnet18_cifar10_successive_halving.yaml")).read_text())
base["study"].update({"name":STUDY_NAME,"output_dir":str(PERSIST_ROOT),"storage_path":str(PERSIST_ROOT/f"{STUDY_NAME}.db"),"resume":RESUME}); base["mode"]=MODE; base["dataset"].update(dataset_cfg); base["model"]={"name":MODEL}; base["objectives"]=OBJECTIVES; base["constraints"]=CONSTRAINTS; base["cost_rates"]=COST_RATES; base["continuous"]=CONTINUOUS_SETTINGS
base["runtime"].update({"device":resource_plan.device,"concurrent_trials":resource_plan.concurrent_trials,"intraop_threads":resource_plan.intraop_threads,"interop_threads":resource_plan.interop_threads,"workers":resource_plan.workers_per_trial,"memory_reserve_gb":MEMORY_RESERVE_GB})
base["search_space"]={"inline":{s.name:{k:v for k,v in s.to_dict().items() if k not in {"name","source","item"} and v not in (None,[],())} for s in specs}}
CONFIG_PATH.write_text(yaml.safe_dump(base,sort_keys=False)); config=load_study_config(CONFIG_PATH)
measurement=next(m for m in calibration.measurements if m.batch_size==calibration.highest_throughput_batch)
calibration_record={"batch_size":measurement.batch_size,"seconds_per_example":measurement.seconds_per_step/measurement.batch_size,"evaluation_seconds_per_epoch":0.0,"checkpoint_seconds_per_epoch":0.0,"checkpoint_size_mb":0.0,"peak_memory_mb":measurement.peak_allocated_mb or 0.0,"cpu_seconds":0.0,"elapsed_seconds":measurement.seconds_per_step}
estimate=estimate_search(config,train_examples=bundle.train_examples,validation_examples=bundle.validation_examples,calibration_records=[calibration_record],representative_batch_size=measurement.batch_size)
ESTIMATE_PATH=PERSIST_ROOT/f"{STUDY_NAME}-pre_search_estimate.json"; ESTIMATE_PATH.write_text(json.dumps(estimate.to_dict(),indent=2)); print(json.dumps(estimate.to_dict(),indent=2))

## 11. Explicit launch gate and Search execution with live monitoring

Review the estimate above. The expensive search starts only after changing `START_SEARCH` to `True`.

In [ ]:
START_SEARCH=False
if START_SEARCH:
    from hpo.monitoring import run_command_with_monitor
    study_dir=PERSIST_ROOT/STUDY_NAME
    log_path=PERSIST_ROOT/f"{STUDY_NAME}.log"
    command=[
        sys.executable,"-m","hpo.cli","--repo-root",str(REPO_ROOT),
        "search","--config",str(CONFIG_PATH),"--mode",MODE,
    ]
    if CONTINUOUS:
        command.append("--continuous")
    expected_records = None if CONTINUOUS else (
        config.full.trials * (len(config.full.budget.seeds)+1)
        if MODE=="full"
        else config.proxy.trials
    )
    return_code=run_command_with_monitor(
        command,cwd=CV_DIR,study_dir=study_dir,log_path=log_path,
        interval_seconds=15,timeout_seconds=None,expected_records=expected_records,
    )
    print("Return code:",return_code)
    if return_code!=0:
        print("Log tail:")
        print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-120:]))
        raise RuntimeError("Search failed; inspect the log above")
else:
    print("Search not started. Set START_SEARCH=True after reviewing the estimate.")

## 12. Resume after reconnection

In [ ]:
# Reconnect, remount Drive if used, rerun setup/config cells, and execute this cell.
RESUME_SEARCH=False
if RESUME_SEARCH:
    from hpo.study import HpoStudy
    resumed=HpoStudy(CONFIG_PATH,repo_root=REPO_ROOT).run()
    print(json.dumps(resumed,indent=2,default=str))
else: print("Resume disabled. Completed trials are stored in SQLite, JSONL, state JSON, and checkpoints.")

## 13. Results analysis and Pareto selection

In [ ]:
from hpo.persistence import read_jsonl
from hpo.reporting import export_reports
from hpo.selection import highest_accuracy_under_budget, fastest_above_accuracy, lowest_memory_above_accuracy, lowest_cost_within_accuracy_margin, pareto_knee
from hpo.schemas import ObjectiveSpec
study_dir=PERSIST_ROOT/STUDY_NAME
if (study_dir/"trials.jsonl").exists():
    rows=read_jsonl(study_dir/"trials.jsonl"); objectives=[ObjectiveSpec(**o) for o in OBJECTIVES]; report=export_reports(study_dir,objectives)
    completed=[r for r in rows if r.get("status")=="completed"]
    print(report); display(pd.read_csv(study_dir/"all_trials.csv")); display(pd.read_csv(study_dir/"pareto_trials.csv"))
    print("Fastest acceptable:",fastest_above_accuracy(completed,minimum_accuracy=0.80))
    print("Lowest memory acceptable:",lowest_memory_above_accuracy(completed,minimum_accuracy=0.80))
    print("Lowest cost near best:",lowest_cost_within_accuracy_margin(completed,accuracy_margin=0.01))
    print("Pareto knee:",pareto_knee(completed,objectives))
else: print("Run or resume a study first.")

## 14. Benchmark comparison

In [ ]:
from hpo.baselines import load_repository_baselines
from hpo.benchmark import compare_with_reference, proxy_reliability
baselines=[b for b in load_repository_baselines(REPO_ROOT) if b.dataset==DATASET and b.model==MODEL]
print([b.__dict__ for b in baselines])
if baselines and 'completed' in locals() and completed:
    best=max(completed,key=lambda r:r.get("metrics",{}).get("validation_top1",0)); print(json.dumps(compare_with_reference(best,max(baselines,key=lambda b:b.validation_top1 or 0),accuracy_margin=0.01),indent=2,default=str))
print("Important: repository stored baselines evaluated against the final test split during training; HPO uses a deterministic validation split and reserves test for confirmation.")

## 15. Export selected hyperparameters and study artifacts

In [ ]:
EXPORT=True
if EXPORT and (PERSIST_ROOT/STUDY_NAME).exists():
    export_dir=PERSIST_ROOT/f"{STUDY_NAME}-export"; export_dir.mkdir(parents=True,exist_ok=True)
    for name in ["all_trials.csv","pareto_trials.csv","best_validation_configuration.json","pareto_knee_configuration.json","report_summary.json","environment.json","resolved_config.json"]:
        src=PERSIST_ROOT/STUDY_NAME/name
        if src.exists(): shutil.copy2(src,export_dir/name)
    shutil.copy2(CONFIG_PATH,export_dir/CONFIG_PATH.name); shutil.copy2(ESTIMATE_PATH,export_dir/ESTIMATE_PATH.name)
    archive=shutil.make_archive(str(export_dir),"zip",root_dir=export_dir); print("Export:",archive)
else: print("Nothing to export yet.")

## Notes and limitations

- Search estimates are projections, not guarantees.
- Colab GPU assignment and session duration vary at runtime.
- One active trial per GPU is the default; multi-trial one-GPU execution requires measured benefit.
- Multi-objective staged promotion uses the designated primary metric; Pareto selection is applied to completed survivors.
- “Best” means best observed for the declared split, search space, fidelity, objectives, constraints, seeds, and budget.